In [1]:
# NB to make a multi busbar from a set of substations with dbb

In [2]:
from __future__ import annotations

import logging

import numpy as np
import pandapower as pp
import pandas as pd
from pandapower import pandapowerNet
from pandapower.create import _get_index_with_check

from pandapower_env.toolbox.pandapower_tools import (
    ELEMENT_TYPES,
    get_element_type_string,
    get_from_bus_str,
    get_to_bus_str,
)
from pandapower_env.toolbox.topology_helpers import (
    find_all_bus_switch_trees,
    find_bus_switch_tree,
)

logger = logging.getLogger(__name__)
from pandapower.networks import case14, case118, case2869pegase

from pandapower_env.environments.gym_env_pp import BaseEnvPP
from pandapower_env.substation.create_double_busbar_substation import (
    can_convert_to_n_busbar_substation,
    copy_bus,
    create_all_double_busbar_substations,
    create_all_dbb_or_3bbwpst_substations,
    create_n_busbar_substation,
    n_assignable_elements_in_bus,
)

from pandapower_env.substation.plot_double_busbar_substation import (
    create_double_busbar_plotting_net,
    separate_substation_buses_visually,
)
from pandapower_env.toolbox.plotting_helpers import (
    calculate_externals_locations,
    create_bus2bus_geodata,
    label_buses,
    label_lines,
    label_substations,
    plot_all_peripherals,
)

In [3]:
net = case2869pegase()

In [4]:
from pandapower_env.substation.create_double_busbar_substation import (
    create_double_busbar_substation,
)

net = case14()

In [5]:
create_all_double_busbar_substations(net)

In [6]:
print(net.multi_bb_substation)
print(net.switch)

   bus_0  bus_1  b01_switch           connected_buses  \
0      1     14           0  [15, 16, 17, 18, 19, 20]   
1      2     21          13          [22, 23, 24, 25]   
2      3     26          22  [27, 28, 29, 30, 31, 32]   
3      4     33          35      [34, 35, 36, 37, 38]   
4      5     39          46  [40, 41, 42, 43, 44, 45]   
5      8     46          59      [47, 48, 49, 50, 51]   
6     12     52          70          [53, 54, 55, 56]   

   n_busbars_in_substation                            element_type  \
0                        2     [line, line, line, line, load, gen]   
1                        2                 [line, line, load, gen]   
2                        2  [line, line, line, trafo, trafo, load]   
3                        2         [line, line, line, trafo, load]   
4                        2    [line, line, line, trafo, load, gen]   
5                        2        [line, line, trafo, trafo, load]   
6                        2                [line, line

In [7]:
# add PSTs
import pandapower.networks as nw

# Load the IEEE 14-bus case
net = nw.case14()

# Create the simplest PST at bus 1
pst_index = pp.create_transformer_from_parameters(
    net,
    hv_bus=1,
    lv_bus=1,
    sn_mva=100,
    vn_hv_kv=69,
    vn_lv_kv=69,
    vk_percent=1,
    vkr_percent=0,
    pfe_kw=0,
    i0_percent=0,
    shift_degree=0,
)

print(f"Simple PST created with index: {pst_index}")

Simple PST created with index: 5


In [8]:
def add_pst_to_net(net, user_pst_config=None):
    """Add a PST to the network, using defaults for missing parameters"""
    # Your default PST configuration
    default_pst_config = {
        "name": "Querregelung PST",
        "std_type": None,
        "hv_bus": 0,
        "lv_bus": 1,
        "sn_mva": 500.0,
        "vn_hv_kv": 380.0,
        "vn_lv_kv": 380.0,
        "vk_percent": 12.0,
        "vkr_percent": 0.1,
        "pfe_kw": 0.0,
        "i0_percent": 0.0,
        "shift_degree": 10.0,
        "tap_side": "hv",
        "tap_neutral": 0,
        "tap_min": -30,
        "tap_max": 30,
        "tap_step_percent": np.nan,
        "tap_step_degree": 1.0,
        "tap_pos": 0,
        "tap_changer_type": "Ideal",
        "id_characteristic_table": None,
        "tap_dependency_table": False,
        "parallel": 1,
        "df": 1.0,
        "in_service": True,
    }

    # Start with defaults, then update with user values
    if user_pst_config is None:
        user_pst_config = {}

    pst_config = default_pst_config.copy()
    pst_config.update(user_pst_config)

    # Create the PST
    pst_index = pp.create_transformer_from_parameters(
        net,
        name=pst_config["name"],
        std_type=pst_config["std_type"],
        hv_bus=pst_config["hv_bus"],
        lv_bus=pst_config["lv_bus"],
        sn_mva=pst_config["sn_mva"],
        vn_hv_kv=pst_config["vn_hv_kv"],
        vn_lv_kv=pst_config["vn_lv_kv"],
        vk_percent=pst_config["vk_percent"],
        vkr_percent=pst_config["vkr_percent"],
        pfe_kw=pst_config["pfe_kw"],
        i0_percent=pst_config["i0_percent"],
        shift_degree=pst_config["shift_degree"],
        tap_side=pst_config["tap_side"],
        tap_neutral=pst_config["tap_neutral"],
        tap_min=pst_config["tap_min"],
        tap_max=pst_config["tap_max"],
        tap_step_percent=pst_config["tap_step_percent"],
        tap_step_degree=pst_config["tap_step_degree"],
        tap_pos=pst_config["tap_pos"],
        tap_changer_type=pst_config["tap_changer_type"],
        id_characteristic_table=pst_config["id_characteristic_table"],
        tap_dependency_table=pst_config["tap_dependency_table"],
        parallel=pst_config["parallel"],
        df=pst_config["df"],
        in_service=pst_config["in_service"],
    )

    return pst_index
add_pst_to_net(net)


np.int64(6)

In [9]:
net = case14()
create_n_busbar_substation(net, 5, n=3)
print(net.multi_bb_substation)

   bus_0  bus_1  bus_2  b01_switch  b02_switch  b12_switch  \
0      5     14     15           0           1           2   

            connected_buses  n_busbars_in_substation  \
0  [16, 17, 18, 19, 20, 21]                        3   

                           element_type  connected_elements  \
0  [line, line, line, trafo, load, gen]  [7, 8, 9, 2, 4, 2]   

             b0_switches             b1_switches             b2_switches  
0  [3, 6, 9, 12, 15, 18]  [4, 7, 10, 13, 16, 19]  [5, 8, 11, 14, 17, 20]  


In [10]:
net = case14()
create_all_dbb_or_3bbwpst_substations(net, [])
print(net.multi_bb_substation)
print(net.trafo)

   bus_0  bus_1  b01_switch           connected_buses  \
0      1     14           0  [15, 16, 17, 18, 19, 20]   
1      2     21          13          [22, 23, 24, 25]   
2      3     26          22  [27, 28, 29, 30, 31, 32]   
3      4     33          35      [34, 35, 36, 37, 38]   
4      5     39          46  [40, 41, 42, 43, 44, 45]   
5      8     46          59      [47, 48, 49, 50, 51]   
6     12     52          70          [53, 54, 55, 56]   

   n_busbars_in_substation                            element_type  \
0                        2     [line, line, line, line, load, gen]   
1                        2                 [line, line, load, gen]   
2                        2  [line, line, line, trafo, trafo, load]   
3                        2         [line, line, line, trafo, load]   
4                        2    [line, line, line, trafo, load, gen]   
5                        2        [line, line, trafo, trafo, load]   
6                        2                [line, line

In [11]:
net = case14()
create_all_dbb_or_3bbwpst_substations(net, [5])
print(net.multi_bb_substation)
print(net.trafo)

   bus_0  bus_1  bus_2  b01_switch  b02_switch  b12_switch  \
0      1     14   <NA>           0        <NA>        <NA>   
1      2     21   <NA>          13        <NA>        <NA>   
2      3     26   <NA>          22        <NA>        <NA>   
3      4     33   <NA>          35        <NA>        <NA>   
4      5     39     40          46          47          48   
5      8     49   <NA>          73        <NA>        <NA>   
6     12     55   <NA>          84        <NA>        <NA>   

                    connected_buses  n_busbars_in_substation  \
0          [15, 16, 17, 18, 19, 20]                        2   
1                  [22, 23, 24, 25]                        2   
2          [27, 28, 29, 30, 31, 32]                        2   
3              [34, 35, 36, 37, 38]                        2   
4  [41, 42, 43, 44, 45, 46, 47, 48]                        3   
5              [50, 51, 52, 53, 54]                        2   
6                  [56, 57, 58, 59]                    

In [12]:
print(net.trafo)

               name std_type  hv_bus  lv_bus  sn_mva  vn_hv_kv  vn_lv_kv  \
0              None     None      30       6  9900.0     135.0    14.000   
1              None     None      31      52  9900.0     135.0     0.208   
2              None     None      37      44  9900.0     135.0     0.208   
3              None     None       6       7  9900.0      14.0    12.000   
4              None     None       6      53  9900.0      14.0     0.208   
5  Querregelung PST     None      48      47   500.0     380.0   380.000   

   vk_percent  vkr_percent  pfe_kw  ...  tap_max  tap_step_percent  \
0    2070.288          0.0     0.0  ...      NaN               2.2   
1    5506.182          0.0     0.0  ...      NaN               3.1   
2    2494.998          0.0     0.0  ...      NaN               6.8   
3    1743.885          0.0     0.0  ...      NaN               NaN   
4    1089.099          0.0     0.0  ...      NaN               NaN   
5      12.000          0.1     0.0  ...     30.

In [13]:
net.trafo.loc[5, "tap_changer_type"]

'Ideal'

In [14]:
net2 = case2869pegase()
is_pst = net2.trafo["tap_changer_type"] == "Ideal"
print(is_pst)

0      False
1      False
2      False
3      False
4      False
       ...  
526    False
527    False
528    False
529    False
530    False
Name: tap_changer_type, Length: 531, dtype: bool


In [15]:

# Show all columns
pd.set_option("display.max_columns", None)
# Show all rows (you only have 89)
pd.set_option("display.max_rows", None)
# Don’t truncate column content
pd.set_option("display.max_colwidth", None)

print(net.switch)

    bus  element et type  closed                   name  z_ohm  in_ka
0    14        1  b   CB    True  busbar switch, 0 to 1    0.0    NaN
1     1       15  b  LBS    True        bus 1 to line 0    0.0    NaN
2    14       15  b  LBS    True       bus 14 to line 0    0.0    NaN
3     1       16  b  LBS    True        bus 1 to line 2    0.0    NaN
4    14       16  b  LBS    True       bus 14 to line 2    0.0    NaN
5     1       17  b  LBS    True        bus 1 to line 3    0.0    NaN
6    14       17  b  LBS    True       bus 14 to line 3    0.0    NaN
7     1       18  b  LBS    True        bus 1 to line 4    0.0    NaN
8    14       18  b  LBS    True       bus 14 to line 4    0.0    NaN
9     1       19  b  LBS    True        bus 1 to load 0    0.0    NaN
10   14       19  b  LBS    True       bus 14 to load 0    0.0    NaN
11    1       20  b  LBS    True         bus 1 to gen 0    0.0    NaN
12   14       20  b  LBS    True        bus 14 to gen 0    0.0    NaN
13   21        2  b 

In [16]:
import pandapower.networks

net =case14()

print(net.bus)
print(net.trafo)

   name    vn_kv type zone  in_service  max_vm_pu  min_vm_pu  \
0     1  135.000    b  1.0        True       1.06       0.94   
1     2  135.000    b  1.0        True       1.06       0.94   
2     3  135.000    b  1.0        True       1.06       0.94   
3     4  135.000    b  1.0        True       1.06       0.94   
4     5  135.000    b  1.0        True       1.06       0.94   
5     6    0.208    b  1.0        True       1.06       0.94   
6     7   14.000    b  1.0        True       1.06       0.94   
7     8   12.000    b  1.0        True       1.06       0.94   
8     9    0.208    b  1.0        True       1.06       0.94   
9    10    0.208    b  1.0        True       1.06       0.94   
10   11    0.208    b  1.0        True       1.06       0.94   
11   12    0.208    b  1.0        True       1.06       0.94   
12   13    0.208    b  1.0        True       1.06       0.94   
13   14    0.208    b  1.0        True       1.06       0.94   

                                       

In [17]:
def add_pst_to_net(net, user_pst_config=None):
    """Add a PST to the network, using defaults for missing parameters"""
    # Your default PST configuration
    default_pst_config = {
        "name": "Querregelung PST",
        "std_type": None,
        "hv_bus": 0,
        "lv_bus": 1,
        "sn_mva": 500.0,
        "vn_hv_kv": 380.0,
        "vn_lv_kv": 380.0,
        "vk_percent": 12.0,
        "vkr_percent": 0.1,
        "pfe_kw": 0.0,
        "i0_percent": 0.0,
        "shift_degree": 10.0,
        "tap_side": "hv",
        "tap_neutral": 0,
        "tap_min": -30,
        "tap_max": 30,
        "tap_step_percent": np.nan,
        "tap_step_degree": 1.0,
        "tap_pos": 0,
        "tap_changer_type": "Ideal",
        "id_characteristic_table": None,
        "tap_dependency_table": False,
        "parallel": 1,
        "df": 1.0,
        "in_service": True,
    }

    # Start with defaults, then update with user values
    if user_pst_config is None:
        user_pst_config = {}

    pst_config = default_pst_config.copy()
    pst_config.update(user_pst_config)

    # Create the PST
    pst_index = pp.create_transformer_from_parameters(
        net,
        name=pst_config["name"],
        std_type=pst_config["std_type"],
        hv_bus=pst_config["hv_bus"],
        lv_bus=pst_config["lv_bus"],
        sn_mva=pst_config["sn_mva"],
        vn_hv_kv=pst_config["vn_hv_kv"],
        vn_lv_kv=pst_config["vn_lv_kv"],
        vk_percent=pst_config["vk_percent"],
        vkr_percent=pst_config["vkr_percent"],
        pfe_kw=pst_config["pfe_kw"],
        i0_percent=pst_config["i0_percent"],
        shift_degree=pst_config["shift_degree"],
        tap_side=pst_config["tap_side"],
        tap_neutral=pst_config["tap_neutral"],
        tap_min=pst_config["tap_min"],
        tap_max=pst_config["tap_max"],
        tap_step_percent=pst_config["tap_step_percent"],
        tap_step_degree=pst_config["tap_step_degree"],
        tap_pos=pst_config["tap_pos"],
        tap_changer_type=pst_config["tap_changer_type"],
        id_characteristic_table=pst_config["id_characteristic_table"],
        tap_dependency_table=pst_config["tap_dependency_table"],
        parallel=pst_config["parallel"],
        df=pst_config["df"],
        in_service=pst_config["in_service"],
    )

    return pst_index
config = {"hv_bus": 5, "lv_bus": 5}
add_pst_to_net(net, config)
print(net.trafo)

               name std_type  hv_bus  lv_bus  sn_mva  vn_hv_kv  vn_lv_kv  \
0              None     None       3       6  9900.0     135.0    14.000   
1              None     None       3       8  9900.0     135.0     0.208   
2              None     None       4       5  9900.0     135.0     0.208   
3              None     None       6       7  9900.0      14.0    12.000   
4              None     None       6       8  9900.0      14.0     0.208   
5  Querregelung PST     None       5       5   500.0     380.0   380.000   

   vk_percent  vkr_percent  pfe_kw  i0_percent  shift_degree tap_side  \
0    2070.288          0.0     0.0         0.0           0.0       hv   
1    5506.182          0.0     0.0         0.0           0.0       hv   
2    2494.998          0.0     0.0         0.0           0.0       hv   
3    1743.885          0.0     0.0         0.0           0.0     None   
4    1089.099          0.0     0.0         0.0           0.0     None   
5      12.000          0.1   